# Multi-task Loss-weight Analysis

Load the comparison CSV produced by `ml/training/sweep_loss_weights.py` and inspect classification accuracy versus nutrition MAE. The aggregate nutrition MAE is the arithmetic mean of absolute errors over calories, protein, carbohydrates, and fat, so it combines values with different native units; inspect the per-target columns before choosing a weight pair. This notebook does not select or write a winning pair.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


cwd = Path.cwd().resolve()
ml_dir = None
for candidate in (cwd, *cwd.parents):
    if (candidate / "training" / "logs").is_dir() and (candidate / "configs" / "base.yaml").is_file():
        ml_dir = candidate
        break
    if (candidate / "ml" / "training" / "logs").is_dir() and (candidate / "ml" / "configs" / "base.yaml").is_file():
        ml_dir = candidate / "ml"
        break
if ml_dir is None:
    raise FileNotFoundError("Could not locate the ml/ project directory from the current directory or its parents.")

comparison_csv = ml_dir / "training" / "logs" / "loss_weight_comparison.csv"
if not comparison_csv.is_file():
    raise FileNotFoundError(
        f"Comparison CSV not found: {comparison_csv}. Run `python -m ml.training.sweep_loss_weights` from the repository root first."
    )

results = pd.read_csv(comparison_csv)
required_columns = {
    "lambda_cls", "lambda_reg", "val_accuracy", "val_nutrition_mae",
    "val_calories_mae", "val_protein_mae", "val_carbohydrates_mae", "val_fat_mae",
}
missing_columns = required_columns - set(results.columns)
if missing_columns:
    raise ValueError(f"Comparison CSV is missing columns: {sorted(missing_columns)}")
if results.empty:
    raise ValueError(f"Comparison CSV contains no completed sweep runs: {comparison_csv}")

results["weight_pair"] = results.apply(
    lambda row: f"({row['lambda_cls']:g}, {row['lambda_reg']:g})", axis=1
)
display(results.sort_values("val_nutrition_mae"))

In [ ]:
fig, axis = plt.subplots(figsize=(9, 6))
points = axis.scatter(
    results["val_accuracy"],
    results["val_nutrition_mae"],
    c=results["lambda_cls"],
    s=80 + 30 * results["lambda_reg"],
    cmap="viridis",
    edgecolors="black",
    alpha=0.85,
)
for _, row in results.iterrows():
    axis.annotate(
        row["weight_pair"],
        (row["val_accuracy"], row["val_nutrition_mae"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
    )
axis.set(
    title="Validation classification–nutrition trade-off",
    xlabel="Validation classification accuracy (higher is better)",
    ylabel="Validation nutrition MAE (lower is better; mean over four targets)",
)
axis.grid(True, alpha=0.3)
colorbar = fig.colorbar(points, ax=axis)
colorbar.set_label("lambda_cls (marker size also increases with lambda_reg)")
fig.tight_layout()
plt.show()

## Interpreting the sweep

Prefer candidates that remain competitive on both axes rather than maximizing one metric while allowing the other to deteriorate. Check the per-target MAE columns in `results` as well: the aggregate metric can hide poor performance on an individual nutrient. These short runs are for screening only; repeat promising pairs with longer runs and, ideally, more than one seed before updating `training.lambda_cls` and `training.lambda_reg` in `ml/configs/base.yaml`.